
---
title: "16. End-to-end integration"
description: "Run one golden-path suite against the Compose POC now and against the deployed platform in Part II: a single definition of done for both environments."
---

## Outcome

Chapters 02–07 built the platform one stage at a time on the Compose sandbox:
tracked data, reproducible training, batch scoring into the results DB, online
serving behind an exact-version contract, dashboards over both, and an LLM
riding the same machinery. This chapter wires nothing new. It proves the stages
already compose into one path, and it pins the definition of done that Part II
will be held to.

The instrument is a single script, `demo/golden_path.py`. It talks only to
contracts: the MLflow client API, the trigger API, `/readyz`, and the results
schema. Because [Chapter 08](./08-environment-contract.ipynb) kept those
contracts identical across backends, the same script runs green locally today
and becomes the acceptance gate for the Azure port in Chapters 09–13.

## The full golden path

```mermaid
flowchart TD
    DATA["tracked dataset (Blob-backed volume)"]
    TRAIN["train job via runner - Ch 03"]
    VER["MLflow run + registered version"]
    EVAL["evaluation - metrics + results row"]
    PROMOTE["promote.py alias flip - Ch 05"]
    BATCH["batch job - Ch 04<br/>parent/child rows"]
    SERVE["serving :18080<br/>/readyz reports exact version"]
    OBS["dashboard :18000 - Ch 06"]

    DATA --> TRAIN --> VER --> EVAL --> PROMOTE
    PROMOTE --> BATCH --> OBS
    PROMOTE --> SERVE --> OBS
```

Read left to right and you have the order the chapters built it in. An LLM app
([Ch 07](./07-llm-release-artifacts.ipynb)) enters at `registered version`
exactly like any other model: nothing downstream knows or cares that the
artifact wraps a prompt pipeline. Every arrow crosses a process boundary
through a contract from Chapter 08, never through a shared-filesystem hunch or
an environment-specific shortcut.



## One script, both environments

`demo/golden_path.py` walks the whole diagram:

1. Trains a model through the runner and waits for the registered version.
2. Promotes the newest version: `python demo/promote.py --version N`
   (`--backend local` is the default; the alias flip plus a one-container
   redeploy of serving is the whole ceremony).
3. Polls `/readyz` on serving until it reports exactly `N`. A flipped alias
   alone proves nothing; the live process must report the version itself.
4. Scores the batch job against the promoted version.
5. Asserts fresh `SUCCESS` rows exist in the results DB for that run.

Three design rules are worth copying into any acceptance suite. Standard
library only (no client dependencies), so the suite itself never becomes an
environment difference. Every poll distinguishes "still running" from "done",
never assuming success on silence. And `--backend aca` refuses to pretend: it
exits pointing at `deploy/smoke-tests.ps1`, whose phases 0–5 walk the same
steps against Azure once Part II lands.

Two companions keep the suite honest. `promote.py --backend aca` prints, and
only applies with `--execute`, the exact `az containerapp update` so cloud
promotion stays reviewable. And `tools/check_env_contract.py` fails when code
reads an env var no backend provides (today: 17 provided across Compose,
10 documented injectors allow-listed).

## How the pieces connect

Locally there is no deploy step to speak of. `docker compose up -d` brings up
the sandbox: Postgres and MinIO underneath, MLflow at :15000, the runner as the
execution plane on :8090, serving on :18080, dashboard on :18000. Promotion is
deliberately boring: merge `DEMO_MODEL_VERSION=N` into `demo/.env` and recreate
one container. Provenance is verifiable end to end because every artifact
carries its own chain: version → MLflow run → evaluation → git tag.

Part II re-hosts these same services on Azure, and the porting work starts
from this chapter's artifacts. The four-pass deploy planned for `deploy.ps1`
(foundation → MLflow + DB setup → images → everything pinned by digest)
becomes the checklist in [Chapter 11](./11-porting-to-aca.ipynb);
`smoke-tests.ps1` phases mirror this suite's five steps one-for-one. The full
Compose↔Azure mapping table lives in [Chapter 08](./08-environment-contract.ipynb).
The capstone adds one rule on top: a backend counts as done only when this
suite passes against it.



## Acceptance evidence

| Phase | Evidence on Compose (runnable today) | Evidence on Azure (Part II) |
|---|---|---|
| Foundation | `check_env_contract.py` exits 0; `docker compose config` parses | Terraform footprint, least-privilege identities (Ch 10) |
| Training | Registered version with dataset + code lineage | Same contract on ACA Jobs (Ch 11) |
| Batch | Parent/child rows; transient item re-dispatched to terminal | Scheduled jobs arrive with the port (Ch 11) |
| Serving | `/readyz` reports exact version; rollback = promote previous again | Same script, different base URL (Ch 11) |
| Observability | Dashboard lists runs and deep-links MLflow | Grafana surfaces and alerts arrive (Ch 13) |
| LLM | pyfunc version serves and batches through unchanged paths | Credentials move to Key Vault (Ch 13) |

A phase is *done* when its evidence is demonstrated, not when its chapter has
been read. For Part I that reduces to one command: `python demo/golden_path.py`
finishing green. When Part II lands, the same verdict comes from the ACA
backend, and nothing in the suite changes except the base URLs.

## Extension catalog

Consolidated from each chapter's Extensions section, plus the Part II scopes
not yet built:

- **Foundation:** private endpoints/VNet, Azure Policy, multi-environment
  topology, Postgres split (Ch 10).
- **Training:** rich stage-identity chain; distributed/multi-GPU escape hatch
  ([Ch 14](./14-multi-gpu-training.ipynb)).
- **Batch:** broker/backpressure for large fan-out
  ([Ch 15](./15-broker-upgrade.ipynb)); event triggers.
- **Serving:** token/scope auth, HTTP-concurrency autoscaling, LLM budget
  partitioning.
- **Observability:** sampling, cross-plane tracing, SLOs, runbooks, budget
  alerts (Ch 13).
- **Delivery:** CI builds, scans, pushes images and updates definitions via
  OIDC (Ch 12); end-to-end automated promotion once the alias flip is wired.

Everything above stays deferrable precisely because the contracts hold. Run
the suite, watch it stay green while you defer, and the MVP is doing its job.
